# Statistical_Modeling

Working through the book Training Systems Using Python Statistical Modeling by Curtis Miller on OReilly

In [ ]:
import logging
import os
from pathlib import Path
import re
import json
import urllib.request
import shutil
import IPython
import pandas as pd
import pyreadstat
from statistical_modeling.config import DEFAULT_INPUT_PPTX_FILENAME, TRACTOR_SERIES_DICT
from statistical_modeling.helper_modules.audit_log import create_audit_log
from pptx import Presentation
from statistical_modeling.helper_modules.excel_output import create_excel_writer, save_excel_writer, \
    write_sheet
from statistical_modeling.config.logging_config import setup_logging
from statistical_modeling.config.defaults import (
    SERIES_KEYS, CURRENT_QUARTER, QUARTERS_LIST, EXCEL_FILE, AUDIT_LOG_FILE, NOTEBOOK_EXCEL_FILE, SLIDE_NAMES
)
from statistical_modeling.config.paths import PATHS
from statistical_modeling.config.periods import build_period_label_map
from statistical_modeling.config.settings import get_settings, build_period_label
from statistical_modeling.helper_modules.preflight import PreflightPaths, preflight
from statistical_modeling.helper_modules import calculate_average_duration, read_all_quarters
from statistical_modeling.helper_modules.meta_proxy import MetaProxy
from statistical_modeling.slide_updaters.context import RunContext
from statistical_modeling.slide_updaters.runners import run_global_steps, run_series_steps
from statistical_modeling.slide_updaters.results import Results
from statistical_modeling.slide_updaters.run_all import run_updaters
from statistical_modeling.helper_modules.write_meta_data_to_excel import write_meta_data_to_excel



logging.basicConfig(level=logging.INFO)
log = logging.getLogger(__name__)

def get_notebook_name():
    # Method 2: ipykernel + server sessions API (JupyterLab / classic Jupyter)
    try:
        import ipykernel
        kernel_id = re.search(
            r'kernel-(.+)\.json', ipykernel.get_connection_file()
        ).group(1)

        try:
            from jupyter_server import serverapp
            servers = list(serverapp.list_running_servers())
        except ImportError:
            from notebook import notebookapp
            servers = list(notebookapp.list_running_servers())

        for server in servers:
            token = server.get('token', '')
            sep = '&' if '?' in server['url'] else '?'
            url = f"{server['url']}api/sessions{sep}token={token}"
            sessions = json.loads(urllib.request.urlopen(url).read())
            for sess in sessions:
                if sess['kernel']['id'] == kernel_id:
                    return os.path.basename(sess['notebook']['path'])
    except Exception:
        pass

    return None


notebook_name = get_notebook_name()
if notebook_name:
    notebook_stem = Path(notebook_name).stem
    print(f"Notebook: {notebook_name}")
    print(f"Stem:     {notebook_stem}")
else:
    notebook_stem = None
    print("Could not determine notebook name automatically.")


settings = get_settings()

# Create audit log with one line per slide updater in jsonl format
audit_log = create_audit_log(AUDIT_LOG_FILE, CURRENT_QUARTER)

# Load metadata from jsonl files (written at ingest time — no .sav read needed)
meta = MetaProxy.from_period(settings.period)

# Read all parquet files for quarters in QUARTERS_LIST, concat them into a dataframe and expose for use in trends
df_all, df_all_labeled = read_all_quarters()
# Filter out current reporting quarter records for use in slides using only current quarter queries
df = df_all[df_all['_quarter'] == CURRENT_QUARTER].copy()
df_labeled = df_all_labeled[df_all_labeled['_quarter'] == CURRENT_QUARTER].copy()

prs = Presentation(PATHS.template_path(DEFAULT_INPUT_PPTX_FILENAME))

# Create Excel writer — keep open until the final save cell
output_excel_path = PATHS.output_dir / f"{notebook_stem}_notebook.xlsx"
writer = create_excel_writer(output_excel_path)
write_meta_data_to_excel(meta, writer)

# write slide_names dict to output excel file
slide_name_dict = {}
for k, v in SLIDE_NAMES.items():
    slide_number = k + 1
    slide_name_dict[slide_number] = v


if writer is not None:
    write_sheet(
        writer,
        pd.DataFrame.from_dict(slide_name_dict, orient='index'),
        sheet_name="Slide Index to Names"
    )
print(f"Excel output: {output_excel_path}")

## Analysis